# Phase 2 submission evidence

Run this read-only notebook in Databricks after the production full/incremental pipeline runs and the isolated drift fixture run. Set demo_batch_id to the batch printed by 98_prepare_drift_demo.ipynb. The results prove the live table schemas, timestamps, key uniqueness, historical MERGE replay, drift handling, quarantine, and execution logs. Export this notebook as IPython Notebook with Include outputs checked. The pipeline implementation is in 02_raw_to_bronze.ipynb, 03_bronze_to_silver.ipynb, and src/carewatch/.


In [ ]:
import re
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "carewatch")
dbutils.widgets.text("schema", "pipeline")
dbutils.widgets.text("historical_batch_id", "14c281a6e8c6c099c0dbad46")
dbutils.widgets.text("demo_schema", "phase2_demo")
dbutils.widgets.text("demo_batch_id", "")

values = {name: dbutils.widgets.get(name).strip() for name in (
    "catalog", "schema", "historical_batch_id", "demo_schema", "demo_batch_id"
)}
for name in ("catalog", "schema", "demo_schema"):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", values[name]):
        raise ValueError(f"Invalid {name}: {values[name]!r}")
for name in ("historical_batch_id", "demo_batch_id"):
    if not re.fullmatch(r"[A-Za-z0-9_]+", values[name]):
        raise ValueError(f"Set a valid {name}; run the drift fixture first for demo_batch_id")

catalog, schema, demo_schema = (values[name] for name in ("catalog", "schema", "demo_schema"))
historical_batch_id = values["historical_batch_id"]
demo_batch_id = values["demo_batch_id"]
prod = lambda name: f"{catalog}.{schema}.{name}"
demo = lambda name: f"{catalog}.{demo_schema}.{name}"
print("Live Databricks evidence scope:", prod("pipeline_execution_logs"))
print("Historical batch:", historical_batch_id, "| Drift demo batch:", demo_batch_id)


## 1. Physical schemas and row checks

The source code defines StructType schemas before file reads. The output below shows the resulting Bronze and Silver table types. The summary verifies that all eight tables have load timestamps and each Silver entity key is unique.


In [ ]:
print("Bronze Provider Information schema")
spark.table(prod("bronze_nh_provider_info")).printSchema()
print("Silver Facility schema")
spark.table(prod("silver_facility")).printSchema()

bronze_checks = [
    ("bronze_nh_health_deficiencies", None),
    ("bronze_nh_penalties", None),
    ("bronze_nh_provider_info", None),
    ("bronze_nh_mds_quality", None),
]
silver_checks = [
    ("silver_deficiency", "deficiency_key"),
    ("silver_penalty", "penalty_key"),
    ("silver_facility", "facility_key"),
    ("silver_mds_quality", "mds_key"),
]
summary = []
for table_name, key in bronze_checks + silver_checks:
    frame = spark.table(prod(table_name))
    stats = frame.agg(
        F.count("*").alias("rows"),
        F.sum(F.col("load_timestamp").isNull().cast("long")).alias("missing_load_timestamps"),
        *( [F.countDistinct(key).alias("distinct_keys")] if key else [] ),
    ).first().asDict()
    summary.append((table_name, stats["rows"], stats["distinct_keys"] if key else None, stats["missing_load_timestamps"]))
summary_df = spark.createDataFrame(summary, "table_name string, rows long, distinct_keys long, missing_load_timestamps long")
display(summary_df)
assert all(row.missing_load_timestamps == 0 for row in summary_df.collect())
assert all(row.distinct_keys == row.rows for row in summary_df.collect() if row.distinct_keys is not None)
print("PASS: all Bronze/Silver load timestamps are present and all Silver entity keys are unique")


## 2. Historical batch, first MERGE, and idempotent rerun

The same registered past batch is used for the first Silver MERGE and the reprocess run. The log shows the parameter, timestamps, status, and inserted/updated counts.


In [ ]:
history = spark.table(prod("pipeline_execution_logs")).filter(
    (F.col("batch_id") == historical_batch_id) &
    (F.col("pipeline_layer").isin("Raw-to-Bronze", "Bronze-to-Silver"))
)
display(history.select(
    "pipeline_layer", "batch_id", "load_type", "parameter_processed",
    "start_time", "end_time", "status", "rows_read", "rows_inserted", "rows_updated"
).orderBy("start_time"))
silver_history = history.filter(
    (F.col("pipeline_layer") == "Bronze-to-Silver") & (F.col("status") == "SUCCESS")
).collect()
assert any(row.rows_inserted > 0 for row in silver_history), "First MERGE insert is missing"
assert any(
    "reprocess=true" in row.parameter_processed and
    row.rows_inserted == 0 and row.rows_updated == 0
    for row in silver_history
), "No successful no-op MERGE rerun exists for this batch"
facility = spark.table(prod("silver_facility"))
facility_counts = facility.agg(
    F.count("*").alias("silver_rows"),
    F.countDistinct("facility_key").alias("distinct_facility_keys"),
    F.sum(F.col("load_timestamp").isNull().cast("long")).alias("missing_load_timestamps")
)
display(facility_counts)
print("PASS: historical parameterized replay, first MERGE, and no-op MERGE rerun")


## 3. Isolated schema drift and quarantine

The separate demo batch has an added source column and a bad rating value. The expected result is two Bronze rows, one accepted Silver row, and one quarantined row with a SAFE_CAST rule.


In [ ]:
drift = spark.table(demo("schema_drift_log")).filter(F.col("batch_id") == demo_batch_id)
bronze_demo = spark.table(demo("bronze_nh_provider_info")).filter(F.col("_batch_id") == demo_batch_id)
silver_demo = spark.table(demo("silver_facility")).filter(F.col("source_batch_id") == demo_batch_id)
quarantine_demo = spark.table(demo("silver_quarantine")).filter(F.col("source_batch_id") == demo_batch_id)

display(drift.select("drift_type", "column_name", "detail", "load_timestamp"))
display(bronze_demo.agg(
    F.count("*").alias("bronze_rows"),
    F.sum(F.col("phase2_demo_added_column").isNotNull().cast("long")).alias("rows_with_added_column")
))
display(spark.createDataFrame(
    [(silver_demo.count(), quarantine_demo.count())],
    "accepted_silver_rows long, quarantined_rows long"
))
display(quarantine_demo.select("failed_rules", "raw_record").limit(5))
assert drift.filter(
    (F.col("drift_type") == "ADDED_COLUMN") &
    (F.col("column_name") == "phase2_demo_added_column")
).count() >= 1
assert bronze_demo.count() == 2 and silver_demo.count() == 1 and quarantine_demo.count() == 1
assert quarantine_demo.filter(F.array_join("failed_rules", ",").contains("SAFE_CAST")).count() == 1
print("PASS: added column recorded, good row accepted, bad typed value quarantined")


## 4. Full and incremental execution logs

Both pipeline stages must have successful full and incremental entries. The summary also includes CMS acquisition as supporting context.


In [ ]:
logs = spark.table(prod("pipeline_execution_logs"))
coverage = logs.groupBy("pipeline_layer", "load_type", "status").agg(
    F.count("*").alias("runs"),
    F.max("end_time").alias("latest_end")
)
display(coverage.orderBy("pipeline_layer", "load_type", "status"))
for layer in ("Raw-to-Bronze", "Bronze-to-Silver"):
    for load_type in ("full", "incremental"):
        assert logs.filter(
            (F.col("pipeline_layer") == layer) &
            (F.col("load_type") == load_type) &
            (F.col("status").isin("SUCCESS", "QUARANTINED_PARTIAL"))
        ).limit(1).count() == 1, f"Missing successful {layer} {load_type} log"
print("PASS: full and incremental execution logs exist for both pipeline stages")
print("All Phase 2 evidence checks passed.")
